In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from src.pipeline.predictor import (
    TicketPredictor,
    predict_ticket,
    get_predictor
)

print("Imports successful")

In [ ]:
predictor = TicketPredictor().load()
print("Predictor ready")

In [ ]:
result = predictor.predict(
    "I cannot access the shared drive. Getting permission denied error."
)
print(result)
print("\nAs dictionary:")
print(result.to_dict())

In [ ]:
test_tickets = [
    # Fileservice
    "Cannot access shared folder on network drive, permission denied",
    # Active Directory
    "Need to create new user account for employee starting Monday",
    # O365
    "Outlook keeps crashing when I try to open email attachments",
    # Software
    "Application throwing null pointer exception on startup",
    # Computer-Services
    "Office printer not responding, shows offline in print queue",
    # EOL
    "Server WSRV-01 needs to be decommissioned and removed from Nexthink",
    # Support general
    "Employee leaving company, need to revoke all system access",
]

print(f"{'Ticket':<55} {'Category':<22} {'Priority':<10} {'Level'}")
print("-" * 100)

for ticket in test_tickets:
    result = predictor.predict(ticket)
    print(
        f"{ticket[:53]:<55} "
        f"{result.category:<22} "
        f"{result.priority:<10} "
        f"{result.priority_level}/3"
    )

In [ ]:
edge_cases = [
    "",                    # Empty string
    "   ",                 # Whitespace only
    "help",                # Too short
    "???!!!###",           # Punctuation only
    "a" * 500,             # Very long repeated character
    "[TICKET ID] - New Support Ticket received",  # Redacted only
]

print("=== EDGE CASE HANDLING ===\n")
for text in edge_cases:
    result = predictor.predict(text)
    display = repr(text[:30])
    print(f"Input:   {display}")
    print(f"Valid:   {result.is_valid}")
    print(f"Warning: {result.warning}")
    print(f"Result:  {result.category} / {result.priority}")
    print()

In [ ]:
batch_tickets = [
    "Cannot login to VPN from home office",
    "Need software license for Adobe Acrobat",
    "Shared mailbox not syncing with Outlook",
    "New joiner needs Active Directory account",
    "Printer in room 204 showing paper jam error",
    "Old laptop needs to be wiped and decommissioned",
    "Cannot open Excel files shared on network drive",
    "Password reset request for locked account",
]

results = predictor.predict_batch(batch_tickets)

print(f"{'Ticket':<50} {'Category':<22} {'Priority'}")
print("-" * 90)
for result in results:
    print(
        f"{result.raw_text[:48]:<50} "
        f"{result.category:<22} "
        f"{result.priority}"
    )

In [ ]:
# Simulate a queue of incoming tickets
incoming_tickets = pd.DataFrame({
    "ticket_id": range(1001, 1009),
    "submitted_by": [
        "john.smith", "jane.doe", "bob.jones",
        "alice.wang", "carlos.silva", "emma.brown",
        "david.lee", "sarah.miller"
    ],
    "text": batch_tickets
})

print("=== INCOMING TICKET QUEUE ===")
print(incoming_tickets[["ticket_id", "text"]].to_string())

print("\n=== AFTER CLASSIFICATION ===")
classified = predictor.predict_dataframe(incoming_tickets, "text")
display_cols = [
    "ticket_id",
    "text",
    "predicted_category",
    "predicted_priority",
    "priority_level"
]
print(classified[display_cols].to_string())

In [ ]:
# Show how a support team would use this output
classified_sorted = classified.sort_values(
    "priority_level",
    ascending=False
)

print("=== TICKET QUEUE SORTED BY PRIORITY ===\n")
print(f"{'ID':<8} {'Priority':<12} {'Category':<22} {'Ticket'}")
print("-" * 85)

for _, row in classified_sorted.iterrows():
    print(
        f"{row['ticket_id']:<8} "
        f"{row['predicted_priority']:<12} "
        f"{row['predicted_category']:<22} "
        f"{row['text'][:40]}"
    )

In [ ]:
# Test the convenience function - this is what Sprint 8 CLI will use
from src.pipeline.predictor import predict_ticket

result = predict_ticket("User locked out of Active Directory account")
print(result)

In [ ]:
import time

# Test prediction speed on 100 tickets
test_texts = batch_tickets * 13  # 104 tickets
test_texts = test_texts[:100]

start = time.time()
results = predictor.predict_batch(test_texts)
elapsed = time.time() - start

print(f"=== PERFORMANCE ===")
print(f"Tickets processed: {len(results)}")
print(f"Total time:        {elapsed:.3f} seconds")
print(f"Per ticket:        {elapsed/len(results)*1000:.2f} ms")
print(f"Throughput:        {len(results)/elapsed:.0f} tickets/second")